In [ ]:
#Run MILD
!python -m run_hybrid_experiment \
  --data data/data/hard/dataset.csv \
  --outdir out_sim/run_16h_seed42_ryu \
  --horizon 15 \
  --folds 3 --epochs 20 --batch 256 --fp_budget 3 \
  --gate_supervise 0.6 --gate_sparsity 0.002 \
  --distill_alpha 0.7 --temp 2.0 --gate_teacher 0.3 \
  --use_teacher_gate

In [ ]:
# Run LSTM Baseline
!python -m baseline_LSTM \
  --data data/hard/dataset.csv \
  --outdir out_final/hard/baseline_LSTM \
  --folds 11 --epochs 30 --batch 512 --fp_budget 1

In [ ]:
#Run MLP Baseline
!python -m baseline_NEW_MLP \
  --data data/hard/dataset.csv \
  --outdir out_final/hard/baselines_NEW_MLP \
  --folds 11 --epochs 30 --batch 512 --fp_budget 1

In [ ]:
#Run Other Baselines
!python -m baselines_multiintent_gem_corrected \
  --data data/hard/dataset.csv \
  --outdir out_final/hard/baselines_F2 \
  --folds 11 --fp_budget 1

In [ ]:
#Generate plot for MILD's Disambiguation of Co-driftng intents
!python -m plot_disambiguation \
  --outdir out_sim/run_16h_seed42 \
  --fold 3 \
  --events data/hard/events_by_intent.json \
  --title ""

In [ ]:
#Generate Plots for MILD's SHAP and Multi horizon modeling
!python -m multi_horizon_and_shap_V2 \
  --data data/hard/dataset.csv \
  --outdir out_final/operational_intelligence \
  --horizons 25 15 5 \
  --fold 2 --folds 3 \
  --intent telemetry \
  --epochs 20 --batch 256 \
  --fp_budget 3 \
  --gate_supervise 0.6 --gate_sparsity 0.002 \
  --distill_alpha 0.7 --temp 2.0 --gate_teacher 0.3


In [ ]:
#Generate Plots for comparison with Baselines (Results are hardcoded in file "final_plots" from above cells)
val = 18
!python -m final_plots --outdir out_final/final_plots --label-size {val} --tick-size {val} --legend-size {val}

In [ ]:
#Ablation Study
from pathlib import Path
import subprocess
import sys
import json

DATA = "data/hard/dataset.csv"
BASE = Path("out_sim/mild_ablations")
BASE.mkdir(parents=True, exist_ok=True)

def run_one(name, include_teacher_gate, extra_args=None):
    extra_args = extra_args or []
    outdir = BASE / name
    outdir.mkdir(parents=True, exist_ok=True)
    log_path = outdir / "stdout.log"

    cmd = [
        sys.executable, "-u", "-m", "run_hybrid_experiment",
        "--data", DATA,
        "--outdir", str(outdir),
        "--horizon", "15",
        "--folds", "3",
        "--epochs", "20",
        "--batch", "256",
        "--fp_budget", "3",
        "--gate_supervise", "0.6",
        "--gate_sparsity", "0.002",
        "--gate_teacher", "0.3",
        "--distill_alpha", "0.6",
        "--temp", "2.0",
    ]

    if include_teacher_gate:
        cmd.append("--use_teacher_gate")

    cmd.extend(extra_args)

    print(f"\n=== {name} ===")
    print(" ".join(cmd))

    with open(log_path, "w", buffering=1) as log:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(f"[{name}] {line}", end="")
            log.write(line)
        rc = proc.wait()

    if rc != 0:
        raise RuntimeError(f"{name} failed; see {log_path}")

    return outdir

runs = [
    # Baseline
    ("baseline", True, []),

    # Ablation 1
    ("abl1_no_distill", True, ["--distill_alpha", "1.0"]),
    ("abl1_no_gate_supervision", True, ["--gate_supervise", "0.0"]),
    ("abl1_strict_no_teacher_gate", False, []),
    ("abl1_no_decorrelation", True, ["--lambda_decorr", "0.0"]),

    # Ablation 2: alpha sweep
    ("sweep_alpha_0p5", True, ["--distill_alpha", "0.5"]),
    ("sweep_alpha_0p7", True, ["--distill_alpha", "0.7"]),
    ("sweep_alpha_0p9", True, ["--distill_alpha", "0.9"]),
    ("sweep_alpha_1p0", True, ["--distill_alpha", "1.0"]),

    # Ablation 2: w_c sweep
    ("sweep_wc_0p3", True, ["--gate_supervise", "0.3"]),
    ("sweep_wc_0p5", True, ["--gate_supervise", "0.5"]),
    ("sweep_wc_0p7", True, ["--gate_supervise", "0.7"]),
    ("sweep_wc_0p9", True, ["--gate_supervise", "0.9"]),
]

summary = []
for name, use_gate, extra in runs:
    outdir = run_one(name, use_gate, extra)
    metrics = outdir / "metrics.json"
    summary.append({
        "run": name,
        "outdir": str(outdir),
        "metrics_exists": metrics.exists()
    })

with open(BASE / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nAll runs finished.")
print("Summary:", BASE / "summary.json")

In [ ]:
#Ablation Study
from pathlib import Path
import subprocess
import sys
import json

DATA = "data/hard/dataset.csv"
BASE = Path("out_sim/mild_ablations_ryu")
BASE.mkdir(parents=True, exist_ok=True)

def run_one(name, include_teacher_gate, extra_args=None):
    extra_args = extra_args or []
    outdir = BASE / name
    outdir.mkdir(parents=True, exist_ok=True)
    log_path = outdir / "stdout.log"

    cmd = [
        sys.executable, "-u", "-m", "run_hybrid_experiment",
        "--data", DATA,
        "--outdir", str(outdir),
        "--horizon", "15",
        "--folds", "3",
        "--epochs", "20",
        "--batch", "256",
        "--fp_budget", "3",
        "--gate_supervise", "0.6",
        "--gate_sparsity", "0.002",
        "--gate_teacher", "0.3",
        "--distill_alpha", "0.6",
        "--temp", "2.0",
    ]

    if include_teacher_gate:
        cmd.append("--use_teacher_gate")

    cmd.extend(extra_args)

    print(f"\n=== {name} ===")
    print(" ".join(cmd))

    with open(log_path, "w", buffering=1) as log:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(f"[{name}] {line}", end="")
            log.write(line)
        rc = proc.wait()

    if rc != 0:
        raise RuntimeError(f"{name} failed; see {log_path}")

    return outdir

runs = [
    (
        "abl1_no_gate_teacher_kl",
        True,
        ["--gate_teacher", "0.0"]
    ),
]

summary = []
for name, use_gate, extra in runs:
    outdir = run_one(name, use_gate, extra)
    metrics = outdir / "metrics.json"
    summary.append({
        "run": name,
        "outdir": str(outdir),
        "metrics_exists": metrics.exists()
    })

with open(BASE / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nAll runs finished.")
print("Summary:", BASE / "summary.json")

In [ ]:
#Ablation Study
from pathlib import Path
import json
import numpy as np
import pandas as pd

BASE = Path("out_sim/mild_ablations_ryu")

def summarize_run(run_name):
    metrics_path = BASE / run_name / "metrics.json"
    if not metrics_path.exists():
        raise FileNotFoundError(f"Missing: {metrics_path}")

    with open(metrics_path, "r") as f:
        folds = json.load(f)["folds"]

    if not folds:
        raise ValueError(f"No folds found in {metrics_path}")

    intents = list(folds[0]["per_intent"].keys())

    fold_fdr = []
    fold_lt = []
    fold_fp = []
    fold_da = []

    for fold in folds:
        fold_fdr.append(np.mean([fold["per_intent"][k]["detection_rate"] for k in intents]) * 100.0)
        fold_lt.append(np.mean([fold["per_intent"][k]["avg_lead_time"] for k in intents]))
        fold_fp.append(fold["overall"]["fp_per_day"])
        fold_da.append(fold["overall"]["disamb_accuracy"] * 100.0)

    return {
        "FDR": float(np.mean(fold_fdr)),
        "LT": float(np.mean(fold_lt)),
        "FP": float(np.mean(fold_fp)),
        "DA": float(np.mean(fold_da)),
    }

# ---------------- Table 8: Ablation study ----------------
table8_runs = [
    ("MILD (full, proposed)", "baseline"),
    ("w/o Knowledge Distillation (α=1)", "abl1_no_distill"),
    ("w/o Gate Cause Supervision (w_c=0)", "abl1_no_gate_supervision"),
    ("w/o Teacher Aug. in Gate", "abl1_strict_no_teacher_gate"),
    ("w/o Decorrelation (λ_decorr=0)", "abl1_no_decorrelation"),
]

table8_rows = []
for label, run_name in table8_runs:
    s = summarize_run(run_name)
    table8_rows.append({
        "Variant": label,
        "FDR": round(s["FDR"], 2),
        "LT": round(s["LT"], 2),
        "FP": round(s["FP"], 2),
        "DA": round(s["DA"], 2),
    })

df8 = pd.DataFrame(table8_rows)
df8.to_csv(BASE / "table8_ablation.csv", index=False)

# ---------------- Table 9: Hyperparameter sensitivity ----------------
table9_rows = []

alpha_runs = [
    ("α", "0.5", "sweep_alpha_0p5"),
    ("α", "0.7", "sweep_alpha_0p7"),
    ("α", "0.9", "sweep_alpha_0p9"),
    ("α", "1.0 (no distill)", "sweep_alpha_1p0"),
]

wc_runs = [
    ("w_c", "0.3", "sweep_wc_0p3"),
    ("w_c", "0.5", "sweep_wc_0p5"),
    ("w_c", "0.7", "sweep_wc_0p7"),
    ("w_c", "0.9", "sweep_wc_0p9"),
]

for param, value, run_name in alpha_runs + wc_runs:
    s = summarize_run(run_name)
    table9_rows.append({
        "Param": param,
        "Value": value,
        "FDR": round(s["FDR"], 2),
        "LT (min)": round(s["LT"], 2),
        "FP": round(s["FP"], 2),
        "DA": round(s["DA"], 2),
    })

df9 = pd.DataFrame(table9_rows)
df9.to_csv(BASE / "table9_sensitivity.csv", index=False)

print("Saved:")
print(" ", BASE / "table8_ablation.csv")
print(" ", BASE / "table9_sensitivity.csv")
print("\nTable 8 preview:")
print(df8.to_string(index=False))
print("\nTable 9 preview:")
print(df9.to_string(index=False))

In [ ]:
#Ablation Study
from pathlib import Path
import subprocess
import sys
import json

DATA = "data/hard/dataset.csv"
BASE = Path("out_sim/mild_ablations")
BASE.mkdir(parents=True, exist_ok=True)

def run_one(name, include_teacher_gate, extra_args=None):
    extra_args = extra_args or []
    outdir = BASE / name
    outdir.mkdir(parents=True, exist_ok=True)
    log_path = outdir / "stdout.log"

    cmd = [
        sys.executable, "-u", "-m", "run_hybrid_experiment",
        "--data", DATA,
        "--outdir", str(outdir),
        "--horizon", "15",
        "--folds", "3",
        "--epochs", "20",
        "--batch", "256",
        "--fp_budget", "3",
        "--gate_supervise", "0.6",
        "--gate_sparsity", "0.002",
        "--gate_teacher", "0.3",
        "--distill_alpha", "0.6",
        "--temp", "2.0",
    ]

    if include_teacher_gate:
        cmd.append("--use_teacher_gate")

    cmd.extend(extra_args)

    print(f"\n=== {name} ===")
    print(" ".join(cmd))

    with open(log_path, "w", buffering=1) as log:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(f"[{name}] {line}", end="")
            log.write(line)
        rc = proc.wait()

    if rc != 0:
        raise RuntimeError(f"{name} failed; see {log_path}")

    return outdir

runs = [
    # Baseline
    ("baseline", True, []),

    # Ablation 1
    ("abl1_no_distill", True, ["--distill_alpha", "1.0"]),
    ("abl1_no_gate_supervision", True, ["--gate_supervise", "0.0"]),
    ("abl1_strict_no_teacher_gate", False, []),
    ("abl1_no_decorrelation", True, ["--lambda_decorr", "0.0"]),

    # Ablation 2: alpha sweep
    ("sweep_alpha_0p5", True, ["--distill_alpha", "0.5"]),
    ("sweep_alpha_0p7", True, ["--distill_alpha", "0.7"]),
    ("sweep_alpha_0p9", True, ["--distill_alpha", "0.9"]),
    ("sweep_alpha_1p0", True, ["--distill_alpha", "1.0"]),

    # Ablation 2: w_c sweep
    ("sweep_wc_0p3", True, ["--gate_supervise", "0.3"]),
    ("sweep_wc_0p5", True, ["--gate_supervise", "0.5"]),
    ("sweep_wc_0p7", True, ["--gate_supervise", "0.7"]),
    ("sweep_wc_0p9", True, ["--gate_supervise", "0.9"]),
]

summary = []
for name, use_gate, extra in runs:
    outdir = run_one(name, use_gate, extra)
    metrics = outdir / "metrics.json"
    summary.append({
        "run": name,
        "outdir": str(outdir),
        "metrics_exists": metrics.exists()
    })

with open(BASE / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nAll runs finished.")
print("Summary:", BASE / "summary.json")